# Fase 5 — Dashboard interactivo (análisis ChEMBL)

| Campo | Valor |
|---|---|
| **Rol líder** | ML Engineer |
| **Entradas** | Fase 2 (`compounds_features.csv`, `activities_clean.csv`) + Fase 4 (`clustering_summary.json`, `stats_tests.csv`) |
| **Salida** | `outputs/dashboard/*.json`, `viz/static/data/*.json` + app FastAPI en `http://127.0.0.1:8001` |
| **Doc** | [`docs/fases/fase5_dashboard.md`](../docs/fases/fase5_dashboard.md) |
| **Comandos** | `make prepare-dashboard` → `make analisis-viz` |

Dashboard **solo del proyecto de análisis de datos** (EDA + resultados multivariados). No incluye visor GNN, mapas ni GeoJSON.


## 0. Configuración

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

import _bootstrap
from src.paths import PROJECT_ROOT as ROOT, setup_path

setup_path()

ARTIFACTS_DIR = ROOT / "outputs" / "dashboard"
STATIC_DATA_DIR = ROOT / "viz" / "static" / "data"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"
COMPOUNDS_CSV = ROOT / "data" / "processed" / "compounds_features.csv"
ACTIVITIES_CSV = ROOT / "data" / "processed" / "activities_clean.csv"

print(f"ROOT        : {ROOT}")
print(f"compounds   : {'OK' if COMPOUNDS_CSV.exists() else 'FALTA — fase2'}")
print(f"activities  : {'OK' if ACTIVITIES_CSV.exists() else 'FALTA — fase2'}")
print(f"clustering  : {'OK' if (RESULTS_DIR / 'clustering_summary.json').exists() else 'FALTA — fase4 §2-§3'}")
print(f"stats_tests : {'OK' if (RESULTS_DIR / 'stats_tests.csv').exists() else 'FALTA — fase4 §3'}")

ROOT        : C:\Users\mateo\Desktop\PROYECTOS\JIC2026\proyecto analisis
compounds   : OK
activities  : OK
clustering  : OK
stats_tests : OK


## 1. Verificar prerequisitos

Obligatorios: Fase 2 (compuestos + actividades) y Fase 4 §2–§3 (clustering + Kruskal/Dunn). El baseline P6 de la Fase 4 §4 es recomendable pero no bloquea la preparación.

In [2]:
prereqs = [
    ("Fase 2 — compounds_features.csv", COMPOUNDS_CSV, True),
    ("Fase 2 — activities_clean.csv", ACTIVITIES_CSV, True),
    ("Fase 4 — clustering_summary.json", RESULTS_DIR / "clustering_summary.json", True),
    ("Fase 4 — stats_tests.csv", RESULTS_DIR / "stats_tests.csv", True),
    ("Fase 4 — baseline_honest_metrics.csv", RESULTS_DIR / "baseline_honest_metrics.csv", False),
]
df_pre = pd.DataFrame(
    [
        (name, "obligatorio" if required else "opcional", p.exists(), str(p.relative_to(ROOT)))
        for name, p, required in prereqs
    ],
    columns=["Prerequisito", "Tipo", "OK", "Ruta"],
)
df_pre["OK"] = df_pre["OK"].map({True: "OK", False: "FALTA"})
display(df_pre)

missing_required = [n for n, p, req in prereqs if req and not p.exists()]
if missing_required:
    raise FileNotFoundError(
        "Faltan prerequisitos obligatorios:\n  - " + "\n  - ".join(missing_required)
    )

,Prerequisito,Tipo,OK,Ruta
0,Fase 2 — compounds_features.csv,obligatorio,OK,data\processed\compounds_features.csv
1,Fase 2 — activities_clean.csv,obligatorio,OK,data\processed\activities_clean.csv
2,Fase 4 — clustering_summary.json,obligatorio,OK,outputs\chembl\results\clustering_summary.json
3,Fase 4 — stats_tests.csv,obligatorio,OK,outputs\chembl\results\stats_tests.csv
4,Fase 4 — baseline_honest_metrics.csv,opcional,OK,outputs\chembl\results\baseline_honest_metrics...


## 2. Generar artefactos JSON del dashboard

Equivale a `make prepare-dashboard` (`scripts/fase5/prepare_dashboard.py`).

In [3]:
cmd = [sys.executable, str(ROOT / "scripts" / "fase5" / "prepare_dashboard.py")]
print("Ejecutando:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
print(result.stdout[-3000:] if result.stdout else "(sin stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError(f"prepare_dashboard falló (exit {result.returncode})")

Ejecutando: c:\Users\mateo\Desktop\PROYECTOS\JIC2026\.venv\Scripts\python.exe C:\Users\mateo\Desktop\PROYECTOS\JIC2026\proyecto analisis\scripts\fase5\prepare_dashboard.py


=== PreparaciÃ³n dashboard (Fase 5) ===
Artefactos en: C:\Users\mateo\Desktop\PROYECTOS\JIC2026\proyecto analisis\outputs\dashboard
Static data : C:\Users\mateo\Desktop\PROYECTOS\JIC2026\proyecto analisis\viz\static\data
Compuestos  : 107
Mediciones  : 2807



## 3. Inspeccionar artefactos generados

In [4]:
def _row_for_artifact(name, path):
    if path.exists():
        if path.suffix == ".csv":
            preview = f"{len(pd.read_csv(path))} filas"
        else:
            payload = json.loads(path.read_text(encoding="utf-8"))
            preview = str(payload)[:100]
        return {"artefacto": name, "existe": True, "bytes": path.stat().st_size, "preview": preview}
    return {"artefacto": name, "existe": False, "bytes": 0, "preview": "—"}


expected_dashboard = [
    "correlation_pearson.json",
    "baseline_honest.json",
    "pchembl_imputation.json",
    "manifest.json",
    "clustering_summary.json",
    "stats_tests.csv",
]
expected_static = [
    "correlation.json",
    "compounds_profile.json",
    "pca_clusters.json",
    "family_stats.json",
]
rows = []
for name in expected_dashboard:
    rows.append(_row_for_artifact(name, ARTIFACTS_DIR / name))
for name in expected_static:
    rows.append(_row_for_artifact(f"static/{name}", STATIC_DATA_DIR / name))
display(pd.DataFrame(rows))

,artefacto,existe,bytes,preview
0,correlation_pearson.json,True,4232,"{'columns': ['mw_freebase', 'alogp', 'psa', 'h..."
1,baseline_honest.json,True,1018,"{'rows': [{'split': 'filas_KFold_CON_FUGA', 'n..."
2,pchembl_imputation.json,True,64,"{'n_total': 107, 'n_imputed': 0, 'pct_imputed'..."
3,manifest.json,True,1251,"{'project': 'proyecto analisis', 'chembl_rows'..."
4,clustering_summary.json,True,763,"{'best_k': 2, 'silhouette_by_k': {'2': 0.33735..."
5,stats_tests.csv,True,740,7 filas
6,static/correlation.json,True,4232,"{'columns': ['mw_freebase', 'alogp', 'psa', 'h..."
7,static/compounds_profile.json,True,67835,"[{'chembl_id': 'CHEMBL101', 'compound_name': '..."
8,static/pca_clusters.json,True,35802,"{'points': [{'chembl_id': 'CHEMBL101', 'compou..."
9,static/family_stats.json,True,2026,{'kruskal_tests': [{'value_col': 'mw_freebase'...


## 4. Smoke test de carga de datos

In [5]:
from viz.services.dashboard.artifacts import load_chembl, load_correlation

chembl_df = load_chembl()
corr = load_correlation()
manifest = json.loads((ARTIFACTS_DIR / "manifest.json").read_text(encoding="utf-8"))

print(f"Compuestos cargados : {len(chembl_df)}")
print(f"Matriz correlación : {len(corr.get('columns', []))} columnas")
print(f"Manifest filas     : {manifest.get('chembl_rows', '—')}")
assert len(chembl_df) > 0, "load_chembl() devolvió DataFrame vacío"

Compuestos cargados : 107
Matriz correlación : 9 columnas
Manifest filas     : 107


## 5. Smoke test de endpoints (FastAPI TestClient)

Verifica `/health`, `/eda` y `/api/analytics/refresh` sin levantar uvicorn.

In [6]:
from fastapi.testclient import TestClient

from viz.app import app

client = TestClient(app)
health = client.get("/health")
print(f"/health              {health.status_code} {health.json()}")

refresh = client.post("/api/analytics/refresh")
print(f"/api/analytics/refresh {refresh.status_code}")

eda_page = client.get("/eda")
print(f"/eda                   {eda_page.status_code} ({len(eda_page.text)} chars)")

meta = client.get("/api/analytics/chembl/meta")
print(f"/api/.../chembl/meta   {meta.status_code}")

assert health.status_code == 200
assert eda_page.status_code == 200

c:\Users\mateo\Desktop\PROYECTOS\JIC2026\.venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


/health              200 {'status': 'ok', 'project': 'proyecto analisis', 'compound_rows': 107}
/api/analytics/refresh 200
/eda                   200 (1971 chars)
/api/.../chembl/meta   200


## 6. Cómo correr el dashboard interactivo

```bash
make prepare-dashboard   # equivalente a la §2
make analisis-viz        # http://127.0.0.1:8001
```

Rutas del proyecto de análisis:
- `/` → redirige a `/eda`
- `/eda` — exploración ChEMBL (Plotly.js)
- `/api/analytics/chembl/*` — datos para gráficos EDA
- `/api/analytics/clusters/pca`, `/api/analytics/families/stats` — resultados Fase 4

## 7. Criterios de éxito

In [7]:
core_artifacts = expected_dashboard + [f"static/{n}" for n in expected_static]
checks = [
    (
        "Artefactos JSON principales",
        all(
            (ARTIFACTS_DIR / n).exists() if not n.startswith("static/") else (STATIC_DATA_DIR / n.split("/", 1)[1]).exists()
            for n in core_artifacts
        ),
        f"{sum((ARTIFACTS_DIR / n).exists() if not n.startswith('static/') else (STATIC_DATA_DIR / n.split('/', 1)[1]).exists() for n in core_artifacts)}/{len(core_artifacts)}",
    ),
    ("load_chembl() > 0 filas", len(chembl_df) > 0, len(chembl_df)),
    ("Health check 200", health.status_code == 200, health.json()),
    ("Página /eda 200", eda_page.status_code == 200, len(eda_page.text)),
]
df_check = pd.DataFrame(checks, columns=["Criterio", "Pasa", "Detalle"])
df_check["Pasa"] = df_check["Pasa"].map({True: "OK", False: "FALTA"})
display(df_check)

,Criterio,Pasa,Detalle
0,Artefactos JSON principales,OK,10/10
1,load_chembl() > 0 filas,OK,107
2,Health check 200,OK,"{'status': 'ok', 'project': 'proyecto analisis..."
3,Página /eda 200,OK,1971


---
*Fase anterior:* [`fase4_modelado.ipynb`](fase4_modelado.ipynb)